# Zemberek-NLP Turkish Dependency Parser (`ZDEP` v1) — Colab / Kaggle Trainer

Trains a compact, pure-Java Turkish Universal Dependencies v2 (`UD_Turkish-BOUN`) dependency parser (`BiGRU + Deep Biaffine + Chu-Liu/Edmonds MST`) and exports a quantized `.bin.gz` model ready to bundle in `zemberek-dependency`.

**Recommended runtime:** Google Colab **T4 / L4 / A100 GPU** (or Kaggle **T4 x2**).
- **Cells 1–4 (Supervised Baseline + Java Parity):** ~2–5 min on GPU.
- **Cells 5–7 (BERTurk Teacher + Mixed-Corpus Distillation):** ~15–20 min on A100 (~40 min on T4) — lifts test accuracy from ~`76.2% UAS / 67.8% LAS` to ~`80.5% UAS / 73.3% LAS` in a `4.2 MB` pure-Java model.

In [ ]:
#@title 1. Setup Environment (GPU check, Java 21 + Maven, Clone Repo, Drive/Kaggle WorkDir)
import os

!nvidia-smi || true

# Install JDK 21 and Maven if missing
!apt-get -qq update && apt-get -qq install -y openjdk-21-jdk maven > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!java -version

# Install Python NLP deps (PyTorch is preinstalled on Colab/Kaggle)
!pip -q install transformers datasets

REPO_URL = "https://github.com/ahmetaa/zemberek-nlp.git"
BRANCH = "master"  #@param {type:"string"}
if not os.path.exists("zemberek-nlp"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL}
else:
    !cd zemberek-nlp && git pull --ff-only

# Auto-detect Kaggle vs Colab + optional Google Drive persistence
MOUNT_DRIVE = True  #@param {type:"boolean"}
if os.path.exists("/kaggle/working"):
    WORK_DIR = "/kaggle/working/zdep_work"
elif MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/zemberek_dependency"
else:
    WORK_DIR = "/content/zdep_work"

os.makedirs(WORK_DIR, exist_ok=True)
print("Working directory:", WORK_DIR)

In [ ]:
#@title 2. Build Zemberek & Align UD_Turkish-BOUN with Zemberek Morphology
import os

# Compile & install dependency + experiment modules locally, then build runtime classpath
!cd zemberek-nlp && mvn -q -pl dependency,experiment -am install -DskipTests
!cd zemberek-nlp && mvn -q -pl experiment dependency:build-classpath -Dmdep.outputFile=/tmp/zemberek_cp.txt

with open("/tmp/zemberek_cp.txt") as f:
    dep_cp = f.read().strip()
ZEMBEREK_CP = dep_cp + ":zemberek-nlp/experiment/target/classes"
os.environ["ZEMBEREK_CP"] = ZEMBEREK_CP

# Download & align UD_Turkish-BOUN train / dev / test (cached on Drive if already present)
FORCE_REALIGN = False  #@param {type:"boolean"}
!mkdir -p {WORK_DIR}/ud_boun {WORK_DIR}/aligned
for split in ["train", "dev", "test"]:
    aligned_out = f"{WORK_DIR}/aligned/boun_{split}.conllu"
    if os.path.exists(aligned_out) and not FORCE_REALIGN:
        print(f"[cache] Reusing existing {aligned_out}")
        continue
    url = f"https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-BOUN/master/tr_boun-ud-{split}.conllu"
    !curl -sSL {url} -o {WORK_DIR}/ud_boun/tr_boun-ud-{split}.conllu
    !java -cp "$ZEMBEREK_CP" zemberek.dependency.UdConlluAligner \
        --input {WORK_DIR}/ud_boun/tr_boun-ud-{split}.conllu \
        --output {aligned_out} \
        --report {WORK_DIR}/aligned/audit_{split}.txt --top 10

In [ ]:
#@title 3. Train Supervised Baseline ZDEP Student (~3-5 min on GPU)
PRESET = "medium"  #@param ["compact", "medium", "large"]
EPOCHS = 35  #@param {type:"integer"}

!python3 zemberek-nlp/experiment/scripts/dependency/train_student.py \
    --train {WORK_DIR}/aligned/boun_train.conllu \
    --dev   {WORK_DIR}/aligned/boun_dev.conllu \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --preset {PRESET} \
    --epochs {EPOCHS} \
    --out-dir {WORK_DIR}/student_supervised_{PRESET}

In [ ]:
#@title 4. Verify Exact PyTorch <-> Pure-Java Parity & Benchmark Speed in Java
STUDENT_DIR = f"{WORK_DIR}/student_supervised_{PRESET}"

print("--- FP32 Parity Check (expects ~100.0% match vs PyTorch) ---")
!java -cp "$ZEMBEREK_CP" zemberek.dependency.EvaluateDependencyModel \
    --model {STUDENT_DIR}/dep_boun_{PRESET}_fp32.bin.gz \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --expected {STUDENT_DIR}/pytorch_test_preds.conllu \
    --passes 3

print("\n--- INT8 Production Model Evaluation + Live Sample Parse ---")
!java -cp "$ZEMBEREK_CP" zemberek.dependency.EvaluateDependencyModel \
    --model {STUDENT_DIR}/dep_boun_{PRESET}_int8.bin.gz \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --expected {STUDENT_DIR}/pytorch_test_preds.conllu \
    --sentence "Prof. Dr. Fethi Bey saat 14.30'da Türkiye'ye döndü." \
    --passes 3

---
## Phase 4: BERTurk Teacher–Student Distillation (+4 to +5.5 LAS boost!)

Fine-tunes `dbmdz/bert-base-turkish-cased` on `UD_Turkish-BOUN`, builds a diverse ~100k-sentence Turkish corpus (Wikipedia + Turkish C4 web/news/essays + raw texts from `IMST`/`Kenet`/`Tourism`/`Atis`/`FrameNet`), projects it via Zemberek, labels high-confidence trees with the teacher, and distills into the compact pure-Java `ZDEP` student.

In [ ]:
#@title 5. Train BERTurk Deep Biaffine Teacher (~5 min on A100 / ~12 min on T4)
import os

TEACHER_BACKBONE = "dbmdz/bert-base-turkish-cased"  #@param ["dbmdz/bert-base-turkish-cased", "dbmdz/convbert-base-turkish-cased"]
TEACHER_EPOCHS = 12  #@param {type:"integer"}
FORCE_RETRAIN_TEACHER = False  #@param {type:"boolean"}
TEACHER_CKPT = f"{WORK_DIR}/teacher_berturk.pt"

if os.path.exists(TEACHER_CKPT) and not FORCE_RETRAIN_TEACHER:
    print(f"[cache] Reusing existing teacher checkpoint: {TEACHER_CKPT}")
else:
    !python3 zemberek-nlp/experiment/scripts/dependency/train_teacher_distill.py train \
        --train {WORK_DIR}/aligned/boun_train.conllu \
        --dev   {WORK_DIR}/aligned/boun_dev.conllu \
        --test  {WORK_DIR}/aligned/boun_test.conllu \
        --backbone {TEACHER_BACKBONE} \
        --epochs {TEACHER_EPOCHS} \
        --batch-size 24 \
        --out {TEACHER_CKPT}

In [ ]:
#@title 6. Build Diverse Mixed Turkish Corpus, Project via Zemberek & Label with Teacher
import os, urllib.request
from datasets import load_dataset

WIKI_PARAS = 40000  #@param {type:"integer"}
C4_PARAS = 40000  #@param {type:"integer"}
MAX_PROJECTED_SENTS = 120000  #@param {type:"integer"}
MIN_CONF = 0.85  #@param {type:"number"}
LABEL_BATCH_SIZE = 128  #@param {type:"integer"}
FORCE_RELABEL = False  #@param {type:"boolean"}

TEACHER_CKPT = f"{WORK_DIR}/teacher_berturk.pt"
assert os.path.exists(TEACHER_CKPT), f"Teacher checkpoint not found at {TEACHER_CKPT} — run Cell 5 first!"

SILVER_OUT = f"{WORK_DIR}/silver_distill.conllu"
if os.path.exists(SILVER_OUT) and not FORCE_RELABEL:
    print(f"[cache] Reusing existing silver corpus: {SILVER_OUT}")
else:
    mixed_txt = f"{WORK_DIR}/unlabeled_mixed_tr.txt"
    lines = []

    # 1. Turkish Wikipedia prose
    wiki = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True)
    wiki_n = 0
    for row in wiki:
        for p in row["text"].split("\n"):
            p = p.strip()
            if 40 <= len(p) <= 500 and not p.startswith("=="):
                lines.append(p)
                wiki_n += 1
                if wiki_n >= WIKI_PARAS:
                    break
        if wiki_n >= WIKI_PARAS:
            break
    print(f"Collected {wiki_n} Wikipedia paragraphs")

    # 2. Turkish C4 web/news/blogs/essays
    c4 = load_dataset("allenai/c4", "tr", split="train", streaming=True)
    c4_n = 0
    for row in c4:
        for p in row["text"].split("\n"):
            p = p.strip()
            if 40 <= len(p) <= 500:
                lines.append(p)
                c4_n += 1
                if c4_n >= C4_PARAS:
                    break
        if c4_n >= C4_PARAS:
            break
    print(f"Collected {c4_n} C4 Turkish paragraphs")

    # 3. Raw '# text =' sentences from other public Turkish UD treebanks (re-labeled by BOUN teacher)
    ud_urls = [
        "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-IMST/master/tr_imst-ud-train.conllu",
        "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Kenet/master/tr_kenet-ud-train.conllu",
        "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Tourism/master/tr_tourism-ud-train.conllu",
        "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Atis/master/tr_atis-ud-train.conllu",
        "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-FrameNet/master/tr_framenet-ud-train.conllu",
    ]
    ud_n = 0
    for url in ud_urls:
        try:
            text = urllib.request.urlopen(url, timeout=15).read().decode("utf-8")
            for line in text.splitlines():
                if line.startswith("# text ="):
                    s = line[len("# text ="):].strip()
                    if len(s) >= 15:
                        lines.append(s)
                        ud_n += 1
        except Exception as e:
            print("Skipped", url, e)
    print(f"Collected {ud_n} extra Turkish UD raw sentences (total lines: {len(lines)})")

    with open(mixed_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    # Project mixed corpus to UD v2 CoNLL-U via Zemberek
    !java -cp "$ZEMBEREK_CP" zemberek.dependency.UdConlluAligner \
        --project-text {mixed_txt} \
        --segment \
        --max-sentences {MAX_PROJECTED_SENTS} \
        --output {WORK_DIR}/unlabeled_mixed_projected.conllu

    # Label with BERTurk teacher & filter by mean arc confidence
    !python3 zemberek-nlp/experiment/scripts/dependency/train_teacher_distill.py label \
        --checkpoint {TEACHER_CKPT} \
        --input  {WORK_DIR}/unlabeled_mixed_projected.conllu \
        --output {SILVER_OUT} \
        --batch-size {LABEL_BATCH_SIZE} \
        --min-conf {MIN_CONF}

In [ ]:
#@title 7. Train Distilled ZDEP Student & Verify Final Pure-Java INT8 Model
DISTILL_PRESET = "medium"  #@param ["compact", "medium", "large"]
DISTILL_EPOCHS = 15  #@param {type:"integer"}
DISTILL_DIR = f"{WORK_DIR}/student_distilled_{DISTILL_PRESET}"

!python3 zemberek-nlp/experiment/scripts/dependency/train_student.py \
    --train  {WORK_DIR}/aligned/boun_train.conllu \
    --dev    {WORK_DIR}/aligned/boun_dev.conllu \
    --test   {WORK_DIR}/aligned/boun_test.conllu \
    --silver {WORK_DIR}/silver_distill.conllu \
    --silver-weight 0.6 \
    --gold-repeat 3 \
    --preset {DISTILL_PRESET} \
    --epochs {DISTILL_EPOCHS} \
    --out-dir {DISTILL_DIR}

!java -cp "$ZEMBEREK_CP" zemberek.dependency.EvaluateDependencyModel \
    --model {DISTILL_DIR}/dep_boun_{DISTILL_PRESET}_int8.bin.gz \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --expected {DISTILL_DIR}/pytorch_test_preds.conllu \
    --sentence "Prof. Dr. Fethi Bey saat 14.30'da Türkiye'ye döndü." \
    --passes 3

# Download final INT8 model directly to your browser (or grab it from Google Drive / Kaggle Output)
FINAL_MODEL = f"{DISTILL_DIR}/dep_boun_{DISTILL_PRESET}_int8.bin.gz"
print("\nExported INT8 model ready at:", FINAL_MODEL)
try:
    from google.colab import files
    files.download(FINAL_MODEL)
except ImportError:
    pass